# Data Splitting:

### Train/Test/Val Split

In [55]:
from sklearn.model_selection import train_test_split
import pandas as pd

In [56]:
df1 = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\projects_of_guvi\Final_project_Deep_learning\data\final_dataset.csv")
img_dirs = r"C:\Users\jagad\Documents\my_classes\Tasks\projects_of_guvi\Final_project_Deep_learning\data\raw_images"

print("First five rows:")
print(df1.head())
print("-" * 50)

print("Datashape: ", df1.shape)
print("-" * 50)


First five rows:
   image_id                                          image_url label  gender  \
0         1  https://images.unsplash.com/photo-172422561835...  REAL  Female   
1         2  https://images.unsplash.com/photo-172422561873...  REAL    Male   
2         3  https://images.unsplash.com/photo-172422561834...  REAL  Female   
3         4  https://images.unsplash.com/photo-172422561855...  REAL  Female   
4         5  https://images.unsplash.com/photo-172422561823...  REAL    Male   

  age_group  image_exists  
0     18-25          True  
1       50+          True  
2     36-50          True  
3     26-35          True  
4     36-50          True  
--------------------------------------------------
Datashape:  (5557, 6)
--------------------------------------------------


### Test splitting:

In [57]:
train_val_df, test_df = train_test_split(   df1,
                                            test_size= 0.15 ,        
                                            stratify=df1['label'],   
                                            random_state=42
                                        )

print("Train and Val size:", train_val_df.shape)
print("Test size:", test_df.shape)

Train and Val size: (4723, 6)
Test size: (834, 6)


### Train and Val splitting:

In [58]:
train_df, val_df = train_test_split(    train_val_df,                         
                                        test_size= 0.176,             
                                        stratify= train_val_df['label'],     
                                        random_state=42
                                    )

print("Train size:", train_df.shape)
print("Val size:", val_df.shape)

Train size: (3891, 6)
Val size: (832, 6)


In [59]:
print("Train label distribution:")
print(train_df['label'].value_counts(normalize=True))

print("\nValidation label distribution:")
print(val_df['label'].value_counts(normalize=True))

print("\nTest label distribution:")
print(test_df['label'].value_counts(normalize=True))

Train label distribution:
label
REAL    0.501928
FAKE    0.498072
Name: proportion, dtype: float64

Validation label distribution:
label
REAL    0.502404
FAKE    0.497596
Name: proportion, dtype: float64

Test label distribution:
label
REAL    0.502398
FAKE    0.497602
Name: proportion, dtype: float64


### Saving three splits into complete separate csv file:

In [60]:
train_df.to_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\projects_of_guvi\Final_project_Deep_learning\data\train.csv", index=False)
val_df.to_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\projects_of_guvi\Final_project_Deep_learning\data\val.csv", index=False)
test_df.to_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\projects_of_guvi\Final_project_Deep_learning\data\test.csv", index=False)

print("Saved train.csv, val.csv, test.csv")

Saved train.csv, val.csv, test.csv


# Data preprocessing/ Augmentation:

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class DeepfakeDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)                                # how many rows/images total

    def __getitem__(self, idx):
        row = self.df.iloc[idx]                            # get the row at position idx
        img_path = f"{self.img_dir}/{row['image_id']}.jpg"
        image = Image.open(img_path).convert("RGB")        # open image, force RGB (avoid grayscale/RGBA issues)

        label = 1 if row['label'] == 'FAKE' else 0         # convert text label to number (models need numbers, not text)

        if self.transform:
            image = self.transform(image)                  # apply resize/normalize/augmentation

        return image, label

In [62]:
train_dataset = DeepfakeDataset(dataframe=train_df, img_dir= img_dirs, transform=None)
test_dataset = DeepfakeDataset(dataframe=test_df, img_dir= img_dirs, transform=None)
val_dataset = DeepfakeDataset(dataframe=val_df, img_dir= img_dirs, transform=None)


In [63]:
print("Number of training samples:", len(train_dataset))

image, label = train_dataset[0]
print("Label:", label)
print("Image type:", type(image))
print("Image size:", image.size)   # since transform=None right now, this is still a PIL Image

Number of training samples: 3891
Label: 1
Image type: <class 'PIL.Image.Image'>
Image size: (256, 256)


### Transform - Compose:

### For training:

Red channel: mean = 0.485, std = 0.229

Green channel: mean = 0.456, std = 0.224

Blue channel: mean = 0.406, std = 0.225

In [64]:
from torchvision import transforms

train_transform = transforms.Compose([
                                        transforms.Resize((224, 224)),
                                        transforms.RandomHorizontalFlip(),
                                        transforms.RandomRotation(10),
                                        transforms.ToTensor(),
                                        transforms.Normalize( mean=[0.485, 0.456, 0.406], 
                                                              std=[0.229, 0.224, 0.225])
                                    ])


### For test/val:

In [65]:
val_test_transform = transforms.Compose([
                                transforms.Resize((224, 224)),
                                transforms.ToTensor(),
                                transforms.Normalize( mean=[0.485, 0.456, 0.406], 
                                                      std=[0.229, 0.224, 0.225])
                                        ])

In [ ]:
train_dataset = DeepfakeDataset(dataframe= train_df, img_dir= img_dirs, transform= train_transform)
val_dataset = DeepfakeDataset(dataframe= val_df, img_dir= img_dirs, transform= val_test_transform)
test_dataset = DeepfakeDataset(dataframe= test_df, img_dir= img_dirs, transform= val_test_transform)

In [67]:
image, label = train_dataset[0]
print("Label:", label)
print("Image type:", type(image))
print("Image shape:", image.shape)

Label: 1
Image type: <class 'torch.Tensor'>
Image shape: torch.Size([3, 224, 224])


# Data Loader:

In [68]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [69]:
images, labels = next(iter(train_loader))
print("Batch of images shape:", images.shape)
print("Batch of labels shape:", labels.shape)

Batch of images shape: torch.Size([32, 3, 224, 224])
Batch of labels shape: torch.Size([32])
